In [5]:
import openmc
import os
import numpy as np
os.system('rm materials.xml')
openmc.reset_auto_ids()

def shell_vol_calc(outer_radius,inner_radius):
    volume=4/3*np.pi*(outer_radius**3-inner_radius**3)
    return volume
def sphere_cap_vol(r,h):
    return 1/3*np.pi*h**2*(3*r-h)

inner_fuel_r = 48
outer_fuel_r = 50
inner_moderator_height = 84

outer_moderator_r = 87
blanket_or = 100


blanket_volume = shell_vol_calc(blanket_or,outer_moderator_r)
moderator_volume = shell_vol_calc(inner_fuel_r,0)-sphere_cap_vol(inner_fuel_r,inner_moderator_height)\
    +shell_vol_calc(outer_moderator_r,outer_fuel_r)
fuel_volume = shell_vol_calc(outer_fuel_r,inner_fuel_r)

def shell_vol_calc(outer_radius,inner_radius):
    volume=4/3*np.pi*(outer_radius**3-inner_radius**3)
    return volume

# Helper: make Li7-enriched lithium fluoride
LiF = openmc.Material(name='LiF')
LiF.add_elements_from_formula(formula='LiF')

LiF_CA = openmc.Material(name='LiF_CA') #LiF with pure Li7
LiF_CA.add_elements_from_formula(formula='LiF',percent_type='ao',enrichment=99.999,
                                 enrichment_target='Li7',enrichment_type='ao')

BeF2 = openmc.Material(name='BeF2')
BeF2.add_elements_from_formula(formula='BeF2')

ZrF4 = openmc.Material(name='ZrF4')
ZrF4.add_elements_from_formula('ZrF4')  # 1 Zr : 4 F

ThF4 = openmc.Material(name='ThF4')
ThF4.add_elements_from_formula('ThF4')  # 1 Th : 4 F

UF4 = openmc.Material(name='UF4')
UF4.add_elements_from_formula('UF4',percent_type='ao',enrichment=4.95
                                 ,enrichment_type='wo')
    #UF4 eneriched to 4.95 % (LEU) and ao means by atom percent



#Define Copenhagen Atomic (CA) fuel salt
fuel_CA_1 = openmc.Material.mix_materials([LiF_CA,UF4,ThF4],[0.73,0.23,0.04],'ao',name="Fuel_CA_1") #from CA_1
fuel_CA_1.volume = fuel_volume
fuel_CA_1.depletable = True


fuel_CA_2 = openmc.Material.mix_materials([LiF_CA,UF4],[0.73,0.27],'ao',name="Fuel_CA_2") #from CA_2 table 1
fuel_CA_2.set_density('g/cm3',4.905)
fuel_CA_2.volume = fuel_volume #245700 in cm^3 from CA-2 table 2
fuel_CA_2.temperature = 626.8 + 273.15 # in Kelvin
fuel_CA_2.depletable = True

#Define Copenhagen Atomic blanket salt
Blanket_CA = openmc.Material.mix_materials([LiF,ThF4],[0.70,0.30],'ao',name="Blanket_CA")
#from CA-2 table 1
Blanket_CA.set_density('g/cm3',4.626)
Blanket_CA.volume = blanket_volume # in cm^3 from CA-2
Blanket_CA.temperature = 626.8 + 273.15 # in Kelvin
Blanket_CA.depletable = True

# Define Fuel A
print(sum([0.64969,0.292,0.05,0.00831]))
fuel_A = openmc.Material.mix_materials([LiF,BeF2,ZrF4,ThF4,UF4]
                                       ,[0.69987,0.237,0.05,0.01,0.00313],'ao',name='Fuel_A')
fuel_A.set_density('kg/m3',2314.668)
fuel_A.depletable = False

# Fuel B
fuel_B = openmc.Material.mix_materials([LiF,BeF2,ZrF4,UF4],
                                       [0.66811,0.29,0.04,0.00189],'ao',name='Fuel_B')
fuel_B.set_density('kg/m3',2154.4833)
fuel_B.depletable = False

# Fuel C
fuel_C = openmc.Material.mix_materials([LiF,BeF2,ZrF4,UF4],
                                        [0.64969,0.292,0.05,0.00831],'ao',name='Fuel_C')
fuel_C.set_density('kg/m3', 2285.8347)
fuel_C.depletable = False

#Definition of air
N2 = openmc.Material(name='N2')
N2.add_elements_from_formula(formula='N2')

O2 = openmc.Material(name='O2')
O2.add_elements_from_formula(formula='O2')

Ar = openmc.Material(name='Ar')
Ar.add_elements_from_formula(formula='Ar')

CO2 = openmc.Material(name='CO2') #add_elements adds C0 so needed to be done "by hand"
#CO2.add_nuclide('C12', 1.0)
#CO2.add_nuclide('O16', 2.0)
CO2.add_elements_from_formula(formula='CO2') 

#With CO2
#air = openmc.Material.mix_materials([N2,O2,Ar,CO2],[0.7808,0.2095,0.0093,0.0004],'vo',name='Air')
#without CO2

air = openmc.Material.mix_materials([N2,O2,Ar],[0.7811,0.2096,0.0093],'vo',name='Air')



# Water 
water = openmc.Material(name="h2o")
water.add_nuclide('H1', 2.0)
water.add_nuclide('O16', 1.0)
water.set_density('g/cm3', 0.75)

water.add_s_alpha_beta('c_H_in_H2O')

# heavy_water 
heavy_water = openmc.Material(name="d2o")
heavy_water.add_nuclide('H2', 2.0)
heavy_water.add_nuclide('O16', 1.0)

heavy_water.set_density('g/cm3', 1.104) #from CA 2 table 1
heavy_water.volume = moderator_volume #2866100 in cm^3
heavy_water.temperature = 26.8 + 273.15 #in Kelvin
heavy_water.depletable = True

heavy_water.add_s_alpha_beta('c_D_in_D2O')


#UO2
uo2 = openmc.Material(name= "uo2",temperature=1200) #Available temperatures are 250, 294, 600, 900, 1200, 2500 K
# Add nuclides to uo2
uo2.add_nuclide('U235', 0.30)
uo2.add_nuclide('U238', 0.70)
uo2.add_nuclide('O16', 2.0)
uo2.set_density('g/cm3', 10.5)
uo2.volume = 1.0e4 
uo2.depletable = True


# Collect and export
materials = openmc.Materials([fuel_A, fuel_B, fuel_C, fuel_CA_1,fuel_CA_2, Blanket_CA, heavy_water, water,air, uo2])
materials.export_to_xml()
print(air)
print("✅ materials.xml has been created")

1.0
Material
	ID             =	17
	Name           =	Air
	Temperature    =	None
	Density        =	24.352839175602057 [g/cm3]
	Volume         =	None [cm^3]
	Depletable     =	False
	S(a,b) Tables  
	Nuclides       
	N14            =	0.7782388307 [ao]
	N15            =	0.0028611693000000003 [ao]
	O16            =	0.20910127776 [ao]
	O17            =	7.94384e-05  [ao]
	O18            =	0.00041928384 [ao]
	Ar36           =	3.1024799999999995e-05 [ao]
	Ar38           =	5.849699999999999e-06 [ao]
	Ar40           =	0.009263125499999999 [ao]

✅ materials.xml has been created
